ベース：地形地図 ESRI World Terrain Base

国境線：HDX OCHA "Syrian Arab Republic – Subnational Administrative Boundaries"
https://data.humdata.org/dataset/cod-ab-syr
syr_admin0.geojson

地域線：HDX OCHA "Syrian Arab Republic – Subnational Administrative Boundaries"
https://data.humdata.org/dataset/cod-ab-syr
syr_admin1.geojson

河川ライン：HDX Humanitarian OpenStreetMap Team "Syria Waterways"
https://data.humdata.org/dataset/hotosm_syr_waterways
syr_rivers_3857.geojson

分析手法：GeoPandas Buffer

対象地域：
・Syria (National Scale)

その他：
・河川ラインをEPSG:3857で処理し、10km bufferを作成
・Folium表示用にEPSG:4326へ再変換
・河川周辺10km圏を地図上に可視化

In [1]:
# Core Libraries
# 地図作成とベクタ処理に必要なライブラリを読み込む

import folium

import geopandas as gpd

/Users/marisa/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# ETL Extract
# 河川ラインデータを読み込む
# CRSがEPSG:3857（メートル単位）か確認する

rivers = gpd.read_file("syr_rivers_3857.geojson")

print(rivers.crs)

EPSG:3857


In [3]:
# ETL Transform
# 河川ラインから10km Bufferを作成
# 河川から10km以内の範囲をPolygonとして生成する

river_buffer = rivers.buffer(10000)

print(type(river_buffer))

print(river_buffer.head())

<class 'geopandas.geoseries.GeoSeries'>
0    POLYGON ((4627931.847 4445725.708, 4628106.845...
1    POLYGON ((3985448.683 4231103.098, 3984625.744...
2    POLYGON ((3980114.552 4229542.348, 3979541.379...
3    POLYGON ((3988846.982 3953079.026, 3988848.125...
4    POLYGON ((4688927.298 4468408.143, 4689908.604...
dtype: geometry


In [4]:
# ETL Transform
# buffer結果をGeoDataFrameへ変換
# CRS情報を保持したまま後続処理に利用する

river_buffer_gdf = gpd.GeoDataFrame(
    geometry=river_buffer,
    crs=rivers.crs
)

print(river_buffer_gdf.crs)

EPSG:3857


In [5]:
# ETL Transform
# Folium表示用（＝緯度経度）にEPSG:4326へ変換
# Buffer作成後のPolygonをWeb地図表示用に再投影する

river_buffer_4326 = river_buffer_gdf.to_crs("EPSG:4326")

print(river_buffer_4326.crs)

EPSG:4326


In [6]:
# Map Setup
# 地形地図ベースを作成

tiles_terrain = 'https://server.arcgisonline.com/ArcGIS/rest/services/World_Terrain_Base/MapServer/tile/{z}/{y}/{x}'
attr_terrain = 'Tiles © Esri — Source: Esri, USGS, NOAA'

m = folium.Map(location=[34.8, 38.5], zoom_start=7, tiles=tiles_terrain, attr=attr_terrain)

In [7]:
# ETL Load
# 河川10km Bufferを地図へ追加

folium.GeoJson(
    river_buffer_4326,
    name='River Buffer 10km',
    style_function=lambda x: {
        'fillColor': '#90e0ef', 
        'color': '#0077b6', 
        'weight': 0.5,            # 縁取りを極細にして、バッファ同士の重なりを美しく
        'fillOpacity': 0.3        # 透明度を少しだけ下げて、背景の山の陰影を透かす
    }
).add_to(m)

In [8]:
# ETL Transform
# 河川ラインをFolium表示用にEPSG:4326へ変換

rivers_4326 = rivers.to_crs("EPSG:4326")

# ETL Load
# Bufferの中心となる河川ラインを地図へ追加

folium.GeoJson(
    rivers_4326,
    name='Rivers',
    style_function=lambda x: {
        'color': '#03045e',
        'weight': 1.2, 
        'opacity': 0.85
    }
).add_to(m)

In [9]:
# ETL Load
# 国境線を地図へ追加
folium.GeoJson('syr_admin0.geojson', name='Country',
               style_function=lambda x: {'color': 'gray', 'weight': 1, 'fillOpacity': 0}).add_to(m)

# ETL Load
# Governorate境界線を地図へ追加
folium.GeoJson('syr_admin1.geojson', name='Governorate',
               style_function=lambda x: {'color': 'gray', 'weight': 1, 'fillOpacity': 0}).add_to(m)

In [10]:
# ETL Load
# 隣国ラベルを地図へ追加

neighbors = {
    "TÜRKIYE": [37.5, 37.5], 
    "IRAQ": [34.5, 42.0],         
    "JORDAN": [31.9, 36.5],       
    "LEBANON": [34.2, 35.0]       
}

for name, coords in neighbors.items():
    folium.Marker(
        location=coords,
        icon=folium.DivIcon(
            html=f'''<div style="font-size: 14pt; font-weight: bold; color: gray; white-space: nowrap; text-align: center; width: 100px; margin-left: -50px;">{name}</div>'''
        )
    ).add_to(m)

In [11]:
# ETL Load
# Governorateラベルを地図へ追加

governorates = {
    "Aleppo": [36.2, 37.5],
    "Al-Hasakeh": [36.5, 40.7],
    "Ar-Raqqa": [36.0, 39.0],
    "As-Sweida": [32.8, 36.9],
    "Daraa": [32.9, 36.2],
    "Deir-ez-Zor": [35.1, 40.5],
    "Damascus": [33.7, 36.7],
    "Hama": [35.2, 37.0],
    "Homs": [34.5, 38.3],
    "Idleb": [35.8, 36.7],
    "Lattakia": [35.6, 36.1],
    "Quneitra": [33.1, 35.9],
    "Rural Damascus": [33.5, 37.5],
    "Tartous": [34.9, 36.1]
}

for name, coords in governorates.items():
    folium.Marker(
        location=coords,
        icon=folium.DivIcon(
            html=f'''<div style="
                font-size: 10pt; 
                color: black; 
                font-weight: bold;
                white-space: nowrap;
                text-align: center;
                width: 100px;
                margin-left: -50px;
                ">{name}</div>'''
        )
    ).add_to(m)

In [12]:
# ETL Load
# HTMLマップとして保存

m.save('11_syria_vector_buffer.html')